Data Cleaning and Preprocessing Summary:
The date column was converted to Datetime format and sorted chronologically from oldest to newest to preserve proper time-series ordering. Missing values and duplicate rows or timestamps were inspected; if duplicate timestamps occur, duplicates are removed while keeping the first observation (keep="first"), and if missing numeric values occur, they are imputed using time interpolation followed by ffill/bfill at the edges. To prevent future data leakage, the dataset is split chronologically into Train, Validation, and Test sets without shuffling, StandardScaler is fit on the Train set only and then applied to Val and Test, and sliding windows of 96 past steps are constructed to predict 96 future steps.

In [9]:
import numpy as np
# Removed: from src.data import prepare_datasets (Functions are defined locally in the notebook)

X_train, Y_train, X_val, Y_val, X_test, Y_test, scaler, report = prepare_datasets(
    uci_id=851,
    train_ratio=0.70,
    val_ratio=0.15,
    input_len=96,
    horizon_len=96,
)

print("=== 1. Cleaning Summary ===")
for k, v in report["clean_summary"].items():
    print(f"  {k}: {v}")

print("\n=== 2. Train / Validation / Test Sizes & Date Ranges ===")
total_rows = sum(report["split_sizes"].values())
for split_name, size in report["split_sizes"].items():
    start_dt, end_dt = report["split_ranges"][split_name]
    print(f"  {split_name.capitalize():5s}: {size:5d} ({size/total_rows:.1%}) | {start_dt} -> {end_dt}")

print("\n=== 3. Sequence Shapes (96 past -> 96 future) ===")
for name, shape in report["shapes"].items():
    print(f"  {name:7s}: {shape}")

print("\n=== 4. Proof: Scaler Fitted on Train ONLY ===")
proof = report["scaler_proof"]
print("  Scaler mean_ (from Train):      ", np.round(proof["scaler_mean"], 4))
print("  Scaler scale_ (from Train):     ", np.round(proof["scaler_scale"], 4))
print("  Train scaled mean (approx 0):   ", np.round(proof["train_scaled_mean"], 4))
print("  Train scaled std  (approx 1):   ", np.round(proof["train_scaled_std"], 4))
print("  Val scaled mean (not forced 0): ", np.round(proof["val_scaled_mean"], 4))
print("  Test scaled mean (not forced 0):", np.round(proof["test_scaled_mean"], 4))

=== 1. Cleaning Summary ===
  initial_missing_values: 0
  exact_duplicate_rows: 0
  duplicate_timestamps_removed: 0
  final_missing_values: 0

=== 2. Train / Validation / Test Sizes & Date Ranges ===
  Train: 24528 (70.0%) | 2018-01-01 00:00:00 -> 2018-09-13 11:45:00
  Val  :  5256 (15.0%) | 2018-09-13 12:00:00 -> 2018-11-07 05:45:00
  Test :  5256 (15.0%) | 2018-11-07 06:00:00 -> 2018-12-31 23:45:00

=== 3. Sequence Shapes (96 past -> 96 future) ===
  X_train: (24337, 96, 7)
  Y_train: (24337, 96)
  X_val  : (5065, 96, 7)
  Y_val  : (5065, 96)
  X_test : (5065, 96, 7)
  Y_test : (5065, 96)

=== 4. Proof: Scaler Fitted on Train ONLY ===
  Scaler mean_ (from Train):       [2.86940000e+01 1.32170000e+01 3.96050000e+00 1.22000000e-02
 8.08242000e+01 8.47862000e+01 4.27077299e+04]
  Scaler scale_ (from Train):      [3.43911000e+01 1.64899000e+01 7.69000000e+00 1.66000000e-02
 1.86238000e+01 3.02328000e+01 2.49401426e+04]
  Train scaled mean (approx 0):    [ 0. -0.  0.  0.  0. -0. -0.]
  Tr